# Aula 03 — ETL KPM → tabela de features

Pipeline de preparação para o projeto (G1–G7):

`JSONL/SQLite (bronze)` → limpeza/tipagem (`silver`) → features por amostra → CSV/Parquet

A avaliação **não** exige Parquet nem Influx; CSV/SQLite bastam se o README explicar a reprodução.


In [1]:
from pathlib import Path
try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x)
import json
import sqlite3
from datetime import datetime, timezone
import pandas as pd

SAMPLE = Path("../datasets/kpm-ue-tp-sample")
DB = SAMPLE / "kpm.sqlite"
OUT = Path("../datasets/kpm-ue-tp-sample/derived")
OUT.mkdir(parents=True, exist_ok=True)

FEATURE_KEYS = [
    "DRB.UEThpUl",
    "DRB.UEThpDl",
    "DRB.RlcSduDelayDl",
    "RRU.PrbTotUl",
]


## 1. Extract — SQLite (+ espelho JSONL opcional)


In [2]:
con = sqlite3.connect(DB)
df = pd.read_sql(
    """
    SELECT run_id, phase, sample_index, ingested_at, source_path, payload_json
    FROM kpm_samples
    ORDER BY phase, sample_index
    """,
    con,
)
con.close()
print(df.shape)
display(df.head(3))


(100, 6)


,run_id,phase,sample_index,ingested_at,source_path,payload_json
0,ue-tp-20260804-174422,baseline,0,2026-08-04T20:44:22.310716+00:00,/home/jakunzler/Documents/GitHub/cesar/cesar-s...,"{""DRB.RlcSduDelayDl"": 55.0, ""DRB.UEThpUl"": 4.4..."
1,ue-tp-20260804-174422,baseline,1,2026-08-04T20:44:22.310716+00:00,/home/jakunzler/Documents/GitHub/cesar/cesar-s...,"{""DRB.RlcSduDelayDl"": 218.0, ""DRB.UEThpUl"": 3...."
2,ue-tp-20260804-174422,baseline,2,2026-08-04T20:44:22.310716+00:00,/home/jakunzler/Documents/GitHub/cesar/cesar-s...,"{""DRB.RlcSduDelayDl"": 137.0, ""DRB.UEThpUl"": 3...."


## 2. Transform — tipagem, nulos, unidade lógica


In [3]:
records = []
for _, row in df.iterrows():
    try:
        payload = json.loads(row["payload_json"])
    except json.JSONDecodeError:
        payload = {}
    rec = {
        "run_id": row["run_id"],
        "phase": row["phase"],
        "sample_index": int(row["sample_index"]),
        "ingested_at": row["ingested_at"],
        "source_path": row["source_path"],
    }
    for key in FEATURE_KEYS:
        val = payload.get(key)
        rec[key] = float(val) if val is not None and val != "" else pd.NA
    records.append(rec)

features = pd.DataFrame.from_records(records)
# ordem didática de fases
phase_order = pd.CategoricalDtype(["baseline", "stress", "recovery"], ordered=True)
features["phase"] = features["phase"].astype(phase_order)
features = features.sort_values(["phase", "sample_index"]).reset_index(drop=True)

print("nulos por coluna:")
display(features.isna().sum())
display(features.head())


nulos por coluna:


run_id                 0
phase                  0
sample_index           0
ingested_at            0
source_path            0
DRB.UEThpUl            0
DRB.UEThpDl          100
DRB.RlcSduDelayDl      0
RRU.PrbTotUl           0
dtype: int64

,run_id,phase,sample_index,ingested_at,source_path,DRB.UEThpUl,DRB.UEThpDl,DRB.RlcSduDelayDl,RRU.PrbTotUl
0,ue-tp-20260804-174422,baseline,0,2026-08-04T20:44:22.310716+00:00,/home/jakunzler/Documents/GitHub/cesar/cesar-s...,4.46,<NA>,55.0,2.0
1,ue-tp-20260804-174422,baseline,1,2026-08-04T20:44:22.310716+00:00,/home/jakunzler/Documents/GitHub/cesar/cesar-s...,3.72,<NA>,218.0,2.0
2,ue-tp-20260804-174422,baseline,2,2026-08-04T20:44:22.310716+00:00,/home/jakunzler/Documents/GitHub/cesar/cesar-s...,3.72,<NA>,137.0,2.0
3,ue-tp-20260804-174422,baseline,3,2026-08-04T20:44:22.310716+00:00,/home/jakunzler/Documents/GitHub/cesar/cesar-s...,3.72,<NA>,171.0,2.0
4,ue-tp-20260804-174422,baseline,4,2026-08-04T20:44:22.310716+00:00,/home/jakunzler/Documents/GitHub/cesar/cesar-s...,3.72,<NA>,39.0,2.0


In [4]:
# Regras simples de qualidade (documente no README do grupo)
# qc -> quality control
from typing import Any


qc = {
    "rows": len(features),
    "phases": features["phase"].astype(str).value_counts().to_dict(),
    "null_fraction": features[FEATURE_KEYS].isna().mean().round(4).to_dict(),
    "generated_at": datetime.now(timezone.utc).isoformat(),
}
# Flags de sanidade
qc["ok_has_baseline_stress"] = {"baseline", "stress"}.issubset(set[Any](features["phase"].astype(str)))
qc["ok_ue_thp_nonneg"] = bool((features["DRB.UEThpUl"].fillna(0) >= 0).all())
print(json.dumps(qc, indent=2))


{
  "rows": 100,
  "phases": {
    "stress": 60,
    "baseline": 20,
    "recovery": 20
  },
  "null_fraction": {
    "DRB.UEThpUl": 0.0,
    "DRB.UEThpDl": 1.0,
    "DRB.RlcSduDelayDl": 0.0,
    "RRU.PrbTotUl": 0.0
  },
  "generated_at": "2026-08-25T02:12:02.097856+00:00",
  "ok_has_baseline_stress": true,
  "ok_ue_thp_nonneg": true
}


## 3. Load — CSV + Parquet (silver local)


In [5]:
csv_path = OUT / "kpm_features.csv"
pq_path = OUT / "kpm_features.parquet"
qc_path = OUT / "etl_qc.json"

features.to_csv(csv_path, index=False)
try:
    features.to_parquet(pq_path, index=False)
    print("Parquet:", pq_path.resolve())
except Exception as exc:
    print("Parquet indisponível (instale pyarrow/fastparquet se quiser):", exc)

qc_path.write_text(json.dumps(qc, indent=2), encoding="utf-8")
print("CSV:", csv_path.resolve())
print("QC:", qc_path.resolve())
display(features.groupby("phase", observed=True)[FEATURE_KEYS].median(numeric_only=True).round(2))


Parquet indisponível (instale pyarrow/fastparquet se quiser): Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
 - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.
CSV: /home/wsl/Documents/learning/data-open-ran/code/datasets/kpm-ue-tp-sample/derived/kpm_features.csv
QC: /home/wsl/Documents/learning/data-open-ran/code/datasets/kpm-ue-tp-sample/derived/etl_qc.json


,DRB.UEThpUl,DRB.RlcSduDelayDl,RRU.PrbTotUl
phase,,,
baseline,3.72,0.0,2.0
stress,80023.68,158.9,99.0
recovery,3.72,0.0,2.0


## 4. Ligação com o lab (opcional)

```bash
# Regenerar bronze a partir do lab
cd ../oai-cn-gnb-nonrt-nearrt
./scripts/run_ue_tp_experiment.sh
```

Próximo: `aula05_inferencia_decisao.ipynb` (MAD / decision / A1 dry-run).
